In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib


df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")

def categorize_sleep(q):
    if q >= 7:
        return "Good"
    elif q >= 5:
        return "Average"
    else:
        return "Poor"

df["SleepQualityLabel"] = df["Quality of Sleep"].apply(categorize_sleep)

X = df.drop(columns=["Person ID", "Quality of Sleep", "SleepQualityLabel"])
y = df["SleepQualityLabel"]

categorical_cols = X.select_dtypes(include=["object"]).columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le  

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("✅ Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

joblib.dump(model, "sleep_quality_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")


✅ Model Accuracy: 0.9866666666666667

Classification Report:
               precision    recall  f1-score   support

     Average       1.00      0.95      0.98        22
        Good       0.98      1.00      0.99        52
        Poor       1.00      1.00      1.00         1

    accuracy                           0.99        75
   macro avg       0.99      0.98      0.99        75
weighted avg       0.99      0.99      0.99        75



['label_encoders.pkl']

In [6]:
print(df["Sleep Disorder"].unique())


[nan 'Sleep Apnea' 'Insomnia']


In [ ]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load("sleep_quality_model.pkl")
scaler = joblib.load("scaler.pkl")
label_encoders = joblib.load("label_encoders.pkl")

new_data = {
    "Gender": ["Male"],
    "Age": [19],
    "Occupation": ["Software Engineer"],
    "Sleep Duration": [4],
    "Physical Activity Level": [40],
    "Stress Level": [8],
    "BMI Category": ["Overweight"],
    "Blood Pressure": ["130/85"],
    "Heart Rate": [80],
    "Daily Steps": [5000],
    "Sleep Disorder": [np.nan] 
}

X_new = pd.DataFrame(new_data)

for col, le in label_encoders.items():
    if col in X_new.columns:
        X_new[col] = X_new[col].fillna(le.classes_[0])
        X_new[col] = le.transform(X_new[col])

X_new_scaled = scaler.transform(X_new)

prediction = model.predict(X_new_scaled)
print("🛌 Predicted Sleep Quality:", prediction[0])


🛌 Predicted Sleep Quality: Average


In [ ]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load("sleep_quality_model.pkl")
scaler = joblib.load("scaler.pkl")
label_encoders = joblib.load("label_encoders.pkl")

gender = input("Enter Gender (Male/Female): ")
age = int(input("Enter Age: "))
occupation = input("Enter Occupation: ")
sleep_duration = float(input("Enter Sleep Duration (hours): "))
physical_activity = int(input("Enter Physical Activity Level (minutes per day): "))
stress_level = int(input("Enter Stress Level (1-10): "))
bmi = input("Enter BMI Category (Normal/Overweight/Obese): ")
bp = input("Enter Blood Pressure (e.g., 120/80): ")
heart_rate = int(input("Enter Heart Rate: "))
daily_steps = int(input("Enter Daily Steps: "))
sleep_disorder = input("Enter Sleep Disorder (Sleep Apnea/Insomnia/leave blank if none): ")

if sleep_disorder.strip() == "":
    sleep_disorder = np.nan   

new_data = pd.DataFrame([{
    "Gender": gender,
    "Age": age,
    "Occupation": occupation,
    "Sleep Duration": sleep_duration,
    "Physical Activity Level": physical_activity,
    "Stress Level": stress_level,
    "BMI Category": bmi,
    "Blood Pressure": bp,
    "Heart Rate": heart_rate,
    "Daily Steps": daily_steps,
    "Sleep Disorder": sleep_disorder
}])

for col, le in label_encoders.items():
    if col in new_data.columns:
        new_data[col] = new_data[col].fillna(le.classes_[0]) 
        new_data[col] = le.transform(new_data[col])

new_data_scaled = scaler.transform(new_data)

prediction = model.predict(new_data_scaled)
print("🛌 Predicted Sleep Quality:", prediction[0])


Enter Gender (Male/Female):  Male
Enter Age:  20
Enter Occupation:  Software Engineer
Enter Sleep Duration (hours):  6
Enter Physical Activity Level (minutes per day):  0
Enter Stress Level (1-10):  9
Enter BMI Category (Normal/Overweight/Obese):  Normal
Enter Blood Pressure (e.g., 120/80):  130/85
Enter Heart Rate:  73
Enter Daily Steps:  4000
Enter Sleep Disorder (Sleep Apnea/Insomnia/leave blank if none):  


🛌 Predicted Sleep Quality: Average


In [ ]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load("sleep_quality_model.pkl")
scaler = joblib.load("scaler.pkl")
label_encoders = joblib.load("label_encoders.pkl")

def predict_from_user():
    print("\n--- Enter Your Details ---")
    
    gender = input("Gender (Male/Female): ")
    age = int(input("Age: "))
    occupation = input("Occupation (e.g., Software Engineer, Doctor, Student): ")
    sleep_duration = float(input("Sleep Duration (hours): "))
    physical_activity = int(input("Physical Activity Level (minutes per day): "))
    stress_level = int(input("Stress Level (1-10): "))
    bmi = input("BMI Category (Normal/Overweight/Obese): ")
    bp = input("Blood Pressure (e.g., 120/80): ")
    heart_rate = int(input("Heart Rate: "))
    daily_steps = int(input("Daily Steps: "))
    sleep_disorder = input("Sleep Disorder (Insomnia/Sleep Apnea/leave blank if none): ")
    
    if sleep_disorder.strip() == "":
        sleep_disorder = "Other"   
    
    user_data = pd.DataFrame([{
        "Gender": gender,
        "Age": age,
        "Occupation": occupation,
        "Sleep Duration": sleep_duration,
        "Physical Activity Level": physical_activity,
        "Stress Level": stress_level,
        "BMI Category": bmi,
        "Blood Pressure": bp,
        "Heart Rate": heart_rate,
        "Daily Steps": daily_steps,
        "Sleep Disorder": sleep_disorder
    }])

    for col, le in label_encoders.items():
        if col in user_data.columns:
            user_data[col] = user_data[col].fillna("Other")
            user_data[col] = user_data[col].apply(lambda val: val if val in le.classes_ else "Other")
            
            if "Other" not in le.classes_:
                le.classes_ = np.append(le.classes_, "Other")
            
            user_data[col] = le.transform(user_data[col])

    user_scaled = scaler.transform(user_data)

    prediction = model.predict(user_scaled)[0]
    print("\n🛌 Predicted Sleep Quality:", prediction)


predict_from_user()



--- Enter Your Details ---


Gender (Male/Female):  Female
Age:  20
Occupation (e.g., Software Engineer, Doctor, Student):  Student
Sleep Duration (hours):  4.5
Physical Activity Level (minutes per day):  20
Stress Level (1-10):  8
BMI Category (Normal/Overweight/Obese):  Obese
Blood Pressure (e.g., 120/80):  130/85
Heart Rate:  75
Daily Steps:  5000
Sleep Disorder (Insomnia/Sleep Apnea/leave blank if none):  



🛌 Predicted Sleep Quality: Poor
